# Drive Me Crazy — 3. PDFormer: Propagation Delay-aware Dynamic Long-range Transformer

A compact re-implementation of PDFormer (Jiang, Han, Zhao, Wang — AAAI 2023,
[arXiv 2301.07945](https://arxiv.org/abs/2301.07945), [official code](https://github.com/BUAABIGSCity/PDFormer)),
written from the paper and the released code, small enough to read in one sitting.

## The idea in three sentences

1. Treat every (time step, sensor) pair as a token, and let tokens attend to each other **across sensors**
   (spatial attention, one attention map per time step) and **across time** (temporal attention, one per
   sensor).
2. Spatial attention is split into two head groups with different sparsity masks: **geographic** heads
   only see sensors within a few road hops; **semantic** heads only see the handful of sensors whose daily
   profile is most similar, wherever they are on the map — the "long-range" part.
3. **Propagation delay**: a sensor's future depends on what its neighbours looked like *a few minutes ago*.
   Instead of the neighbour's current embedding, the geographic keys are enriched with a summary of each
   sensor's **last few readings** matched against a small dictionary of typical short-term traffic patterns
   (learned by clustering the training data). This "delay-aware feature transformation" is the module we
   ablate at the end.

Everything else is a standard pre-LayerNorm transformer encoder with skip connections into a small
output head that maps `T_in` encoded steps to `T_out` predicted ones.

## What is simplified vs. the paper (and why it is safe)

| paper / official code | here | justification |
|---|---|---|
| k-Shape clustering of length-3 windows into 16 patterns | k-means on a 100 k subsample | the released code's own fallback; with 3-step windows shape invariance buys nothing |
| DTW distance between daily profiles for the semantic mask | correlation distance | the paper ablates *mask vs. no mask*, never the distance; one `cdist` call instead of minutes of DTW |
| 6 encoder layers, 200 epochs, curriculum learning, stochastic depth | 4 layers, ≤ 30 epochs, no curriculum, dropout 0.1 | compute budget; the paper's own hyper-parameter search covers depth 2–8 |
| 1 440 minute-of-day embeddings | one embedding per time slot (288 / 48) | identical information at the data's resolution |

Metrics, splits, masks (7 hops / 5 semantic neighbours), Laplacian positional encoding, Huber loss on the
original scale, AdamW + cosine schedule and the evaluation protocol follow the paper.

In [ ]:
import gc, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

DATA, RESULTS = Path("data"), Path("results")
RESULTS.mkdir(exist_ok=True)

# Per-dataset facts (paper: PDFormer, Jiang et al., AAAI 2023, Table 1) and the forecasting setup PDFormer uses.
# min_true: readings below it are treated as missing when scoring (0 = sensor gap; NYC cells with < 10 trips).
DATASETS = {
    "PEMS04":  dict(step_min=5,  start="2018-01-01", channels=["flow"],              min_true=1,  T_in=12, T_out=12, split=(0.6, 0.2, 0.2)),
    "PEMS08":  dict(step_min=5,  start="2016-07-01", channels=["flow"],              min_true=1,  T_in=12, T_out=12, split=(0.6, 0.2, 0.2)),
    "PEMS07":  dict(step_min=5,  start="2017-05-01", channels=["flow"],              min_true=1,  T_in=12, T_out=12, split=(0.6, 0.2, 0.2)),
    "NYCTaxi": dict(step_min=30, start="2014-01-01", channels=["inflow", "outflow"], min_true=10, T_in=6,  T_out=1,  split=(0.7, 0.1, 0.2)),
}
# DMC_DATASETS="PEMS08" restricts a run (used by the Docker smoke test); default = all four.
SELECTED = [d for d in os.environ.get("DMC_DATASETS", ",".join(DATASETS)).split(",") if d]


def grid_adjacency(rows, cols):
    """8-neighbour adjacency of a rows x cols grid; cell index = row * cols + col."""
    r, c = np.divmod(np.arange(rows * cols), cols)
    adj = (np.abs(r[:, None] - r[None]) <= 1) & (np.abs(c[:, None] - c[None]) <= 1)
    np.fill_diagonal(adj, False)
    return adj.astype(np.float32)


def load_dataset(name):
    """X (T, N, C) float32 targets, adj (N, N) binary symmetric, times DatetimeIndex of length T."""
    cfg = DATASETS[name]
    if name.startswith("PEMS"):
        X = np.load(DATA / f"{name}.npz")["data"][..., :1].astype(np.float32)  # channel 0 = traffic flow
        adj = np.load(DATA / f"adj_{name}.npy").astype(np.float32)
    else:
        df = pd.read_csv(DATA / "NYCTaxi" / "NYCTaxi.grid")
        rows, cols, T = df.row_id.max() + 1, df.column_id.max() + 1, df.time.nunique()
        assert df.dyna_id.is_monotonic_increasing  # cell-major file: every timestep of cell 0, then cell 1, ...
        X = df[cfg["channels"]].to_numpy(np.float32).reshape(rows * cols, T, len(cfg["channels"])).transpose(1, 0, 2)
        adj = grid_adjacency(rows, cols)
    times = pd.date_range(cfg["start"], periods=len(X), freq=f"{cfg['step_min']}min")
    return np.ascontiguousarray(X), adj, times


def time_features(times, step_min):
    """Time-of-day slot index and day-of-week for every timestep."""
    tod = ((times.hour * 60 + times.minute) // step_min).to_numpy()
    return tod, times.dayofweek.to_numpy()


def windows(X, T_in, T_out):
    """Zero-copy sliding windows of length T_in + T_out: (n, T_in + T_out, N, C) view. Index it to copy a batch."""
    return np.moveaxis(sliding_window_view(X, T_in + T_out, axis=0), -1, 1)


def split_indices(n_windows, split):
    """Chronological train / val / test window indices."""
    a, b = int(n_windows * split[0]), int(n_windows * (split[0] + split[1]))
    return np.arange(a), np.arange(a, b), np.arange(b, n_windows)


def masked_metrics(pred, true, min_true):
    """MAE / RMSE / MAPE over entries whose ground truth is >= min_true."""
    m = true >= min_true
    err = (pred - true)[m]
    return dict(MAE=np.abs(err).mean(), RMSE=np.sqrt((err ** 2).mean()), MAPE=100 * (np.abs(err) / true[m]).mean())


def metrics_by_horizon(pred, true, min_true):
    """pred / true (n, T_out, N, C) -> one row per horizon plus an 'avg' row (mean of the per-horizon metrics)."""
    rows = [dict(horizon=str(h + 1), **masked_metrics(pred[:, h], true[:, h], min_true)) for h in range(pred.shape[1])]
    df = pd.DataFrame(rows)
    avg = df.drop(columns="horizon").mean().to_dict()
    return pd.concat([df, pd.DataFrame([dict(horizon="avg", **avg)])], ignore_index=True)


def sample_nodes(adj, X):
    """Three nodes to plot: the best-connected sensor, the busiest, and the quietest (non-dead)."""
    mean_flow = X[..., 0].mean(0)
    alive = np.where(mean_flow > np.percentile(mean_flow, 5))[0]
    return sorted({int(adj.sum(0).argmax()), int(mean_flow.argmax()), int(alive[mean_flow[alive].argmin()])})


def sample_frame(times, test_idx, T_in, T_out, nodes, true, preds, n_days, step_min, model_names):
    """Long table (time, node, actual, <model>_h1, <model>_hLast) over the first n_days of the test period."""
    steps = n_days * 24 * 60 // step_min
    rows = []
    for k in range(min(steps, len(test_idx))):
        w = test_idx[k]
        t1, tL = times[w + T_in], times[w + T_in + T_out - 1]
        for n in nodes:
            rec = dict(time_h1=t1, time_hL=tL, node=n, actual_h1=true[k, 0, n, 0], actual_hL=true[k, -1, n, 0])
            for name, p in zip(model_names, preds):
                rec[f"{name}_h1"], rec[f"{name}_hL"] = p[k, 0, n, 0], p[k, -1, n, 0]
            rows.append(rec)
    return pd.DataFrame(rows)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse.csgraph import shortest_path
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUTOCAST = dict(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda")   # bf16 attention: ~2x faster, half the memory
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
torch.manual_seed(0); np.random.seed(0)

## 1. Graph pre-processing: masks, positional encoding, traffic patterns

Four things are computed once per dataset from the **training period only**:

- `geo_mask[i, j]` — True (blocked) when sensor `j` is 7 or more road hops from `i` (on the taxi grid: king-move
  distance ≥ 3, i.e. each cell sees its 5 × 5 neighbourhood). The road graphs are sparse — on PeMS07 a sensor
  sees 13 others on average, 35 at most, out of 883 — so the model stores each sensor's allowed list and
  attends only over it instead of masking a dense N × N map (same result, a fraction of the memory).
- `sem_mask[i, j]` — True unless `j` is among the 5 sensors (self included) whose mean daily profile is most
  correlated with `i`'s.
- `lap_pe` — the 8 smoothest non-trivial eigenvectors of the normalised graph Laplacian, a coordinate system
  for "where on the road network am I" that a transformer otherwise lacks.
- `pattern_keys` — 16 typical 3-step traffic snippets (k-means centroids of standardised windows).

In [ ]:
S_HIST, N_PATTERNS, LAPE_DIM, SEM_K = 3, 16, 8, 5
FAR_HOPS = {"PEMS04": 7, "PEMS07": 7, "PEMS08": 7, "NYCTaxi": 3}


def build_graph_inputs(name, X, adj, train_steps, spd):
    hops = shortest_path(adj, unweighted=True, directed=False)
    geo_mask = hops >= FAR_HOPS[name]                                   # inf (disconnected) counts as far

    days = train_steps // spd
    daily = X[:days * spd, :, 0].reshape(days, spd, -1).mean(0).T       # (N, spd) mean daily profile per node
    sem_mask = np.ones_like(geo_mask)
    nearest = np.argsort(cdist(daily, daily, metric="correlation"), 1)[:, :SEM_K]
    np.put_along_axis(sem_mask, nearest, False, axis=1)

    deg = adj.sum(1)
    d_inv = np.where(deg > 0, deg ** -0.5, 0.0)
    lap = np.eye(len(adj)) - d_inv[:, None] * adj * d_inv[None]
    vals, vecs = np.linalg.eigh(lap)
    lap_pe = vecs[:, vals > 1e-8][:, :LAPE_DIM].astype(np.float32)     # skip the constant eigenvector(s)

    mu, sd = X[:train_steps].mean(), X[:train_steps].std()
    xs = (X[:train_steps] - mu) / sd
    snippets = np.moveaxis(sliding_window_view(xs, S_HIST, axis=0), -1, 1)   # (t, S, N, C)
    snippets = snippets.transpose(0, 2, 1, 3).reshape(-1, S_HIST * X.shape[2])
    sub = snippets[np.random.default_rng(0).choice(len(snippets), min(100_000, len(snippets)), replace=False)]
    keys = KMeans(N_PATTERNS, n_init=3, random_state=0).fit(sub).cluster_centers_.astype(np.float32)
    return dict(geo_mask=geo_mask, sem_mask=sem_mask, lap_pe=lap_pe, pattern_keys=keys, mu=float(mu), sd=float(sd))

## 2. The model

Shapes: `B` batch, `T` input steps, `N` sensors, `C` channels, `D` embedding width (64), `S` = 3 history
steps for the delay module, `K` = 16 patterns.

In [ ]:
def neighbour_lists(keep):
    # (N, N) bool "may attend" -> idx (N, K) neighbour indices padded with self, valid (N, K) padding flags
    K = int(keep.sum(1).max())
    idx = np.repeat(np.arange(len(keep))[:, None], K, 1)
    valid = np.zeros((len(keep), K), bool)
    for i, row in enumerate(keep):
        nb = np.flatnonzero(row)
        idx[i, :len(nb)], valid[i, :len(nb)] = nb, True
    return idx, valid


class STAttention(nn.Module):
    # One layer's attention: temporal heads (over T, per sensor) + geographic and semantic heads (over N, per step).
    def __init__(self, D, heads, use_delay, dropout):
        super().__init__()
        self.hg, self.hs, self.ht = heads
        self.hd = D // sum(heads)
        self.Dg, self.Ds, self.Dt = self.hg * self.hd, self.hs * self.hd, self.ht * self.hd
        self.geo_qkv, self.sem_qkv, self.t_qkv = nn.Linear(D, 3 * self.Dg), nn.Linear(D, 3 * self.Ds), nn.Linear(D, 3 * self.Dt)
        self.use_delay = use_delay
        if use_delay:  # delay-aware feature transformation: attention of each token's recent history over the pattern dictionary
            self.pat_q, self.pat_k, self.pat_v = nn.Linear(D, self.Dg), nn.Linear(D, self.Dg), nn.Linear(D, self.Dg)
        self.proj = nn.Linear(D, D)
        self.drop = nn.Dropout(dropout)

    @staticmethod
    def heads_over(x, h, hd, axis):
        # (B, T, N, h*hd) -> attention over `axis` ("N" or "T"): (B, other, h, axis_len, hd)
        B, T, N, _ = x.shape
        x = x.view(B, T, N, h, hd)
        return x.permute(0, 1, 3, 2, 4) if axis == "N" else x.permute(0, 2, 3, 1, 4)

    def sparse_attention(self, q, k, v, idx, valid):
        # q, k, v (B, T, h, N, hd); idx / valid (N, K): each sensor attends only the K sensors its mask allows.
        # A masked softmax over N is the same number as a softmax over those K, so gathering the allowed keys
        # gives identical output at K/N of the memory and compute (PeMS07: K <= 35 of N = 883).
        kk, vv = k[:, :, :, idx], v[:, :, :, idx]                                    # (B, T, h, N, K, hd)
        scores = torch.einsum("bthnd,bthnkd->bthnk", q, kk) / self.hd ** 0.5
        att = torch.softmax(scores.masked_fill(~valid, float("-inf")), -1)
        return torch.einsum("bthnk,bthnkd->bthnd", att, vv)

    def forward(self, x, pat_h, pat_k, geo, sem):
        B, T, N, D = x.shape
        q, k, v = self.t_qkv(x).chunk(3, -1)
        out_t = F.scaled_dot_product_attention(*(self.heads_over(a, self.ht, self.hd, "T") for a in (q, k, v)))
        out_t = out_t.permute(0, 3, 1, 2, 4).reshape(B, T, N, self.Dt)

        q, k, v = self.geo_qkv(x).chunk(3, -1)
        if self.use_delay:
            pq = self.pat_q(pat_h).view(B, T, N, self.hg, self.hd)
            pk, pv = self.pat_k(pat_k).view(-1, self.hg, self.hd), self.pat_v(pat_k).view(-1, self.hg, self.hd)
            att = torch.softmax(torch.einsum("btnhd,khd->btnhk", pq, pk) / self.hd ** 0.5, -1)
            k = k + torch.einsum("btnhk,khd->btnhd", att, pv).reshape(B, T, N, self.Dg)
        out_g = self.sparse_attention(*(self.heads_over(a, self.hg, self.hd, "N") for a in (q, k, v)), *geo)
        out_g = out_g.permute(0, 1, 3, 2, 4).reshape(B, T, N, self.Dg)

        q, k, v = self.sem_qkv(x).chunk(3, -1)
        out_s = self.sparse_attention(*(self.heads_over(a, self.hs, self.hd, "N") for a in (q, k, v)), *sem)
        out_s = out_s.permute(0, 1, 3, 2, 4).reshape(B, T, N, self.Ds)
        return self.drop(self.proj(torch.cat([out_t, out_g, out_s], -1)))


class EncoderBlock(nn.Module):
    def __init__(self, D, heads, use_delay, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D, eps=1e-6), nn.LayerNorm(D, eps=1e-6)
        self.attn = STAttention(D, heads, use_delay, dropout)
        self.mlp = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * D, D), nn.Dropout(dropout))

    def forward(self, x, pat_h, pat_k, geo, sem):
        x = x + self.attn(self.ln1(x), pat_h, pat_k, geo, sem)
        return x + self.mlp(self.ln2(x))


class PDFormer(nn.Module):
    def __init__(self, N, C, T_in, T_out, spd, graph, D=64, depth=4, heads=(4, 2, 2), skip_dim=256, use_delay=True, dropout=0.1):
        super().__init__()
        self.T_in, self.T_out, self.C, self.use_delay = T_in, T_out, C, use_delay
        self.value = nn.Linear(C, D)
        pos = torch.zeros(T_in, D)
        div = torch.exp(torch.arange(0, D, 2) * (-np.log(10000.0) / D))
        pos[:, 0::2], pos[:, 1::2] = torch.sin(torch.arange(T_in)[:, None] * div), torch.cos(torch.arange(T_in)[:, None] * div)
        self.register_buffer("pos", pos)
        self.tod, self.dow = nn.Embedding(spd, D), nn.Embedding(7, D)
        self.lape = nn.Linear(LAPE_DIM, D)
        self.register_buffer("lap_pe", torch.tensor(graph["lap_pe"]))
        for tag in ("geo", "sem"):                                             # masks -> padded neighbour lists
            idx, valid = neighbour_lists(~graph[f"{tag}_mask"])
            self.register_buffer(f"{tag}_idx", torch.tensor(idx)); self.register_buffer(f"{tag}_valid", torch.tensor(valid))
        if use_delay:
            self.pattern = nn.Linear(S_HIST * C, D)                            # shared by histories and dictionary keys
            self.register_buffer("pattern_keys", torch.tensor(graph["pattern_keys"]))
        self.blocks = nn.ModuleList(EncoderBlock(D, heads, use_delay, dropout) for _ in range(depth))
        self.skips = nn.ModuleList(nn.Linear(D, skip_dim) for _ in range(depth))
        self.time_proj = nn.Linear(T_in, T_out)
        self.out = nn.Linear(skip_dim, C)

    def forward(self, x, tod, dow):
        # x (B, T_in, N, C) standardised; tod / dow (B, T_in) integer slots
        B, T, N, C = x.shape
        h = self.value(x) + self.pos[None, :, None] + self.tod(tod)[:, :, None] + self.dow(dow)[:, :, None] + self.lape(self.lap_pe)[None, None]
        pat_h = pat_k = None
        if self.use_delay:
            hist = torch.stack([F.pad(x, (0, 0, 0, 0, s, 0))[:, :T] for s in reversed(range(S_HIST))], -2)  # (B, T, N, S, C): x[t-S+1..t]
            pat_h, pat_k = self.pattern(hist.reshape(B, T, N, -1)), self.pattern(self.pattern_keys)
        skip = 0
        for block, proj in zip(self.blocks, self.skips):
            h = block(h, pat_h, pat_k, (self.geo_idx, self.geo_valid), (self.sem_idx, self.sem_valid))
            skip = skip + proj(h)
        z = self.time_proj(F.relu(skip).permute(0, 2, 3, 1))                  # (B, N, skip, T_out)
        return self.out(F.relu(z).permute(0, 3, 1, 2))                       # (B, T_out, N, C)

## 3. Training and evaluation

Huber loss (δ = 2) on the de-standardised prediction, AdamW (lr 1e-3, weight decay 0.05), linear warm-up
then cosine decay, gradient clipping at 5, early stopping on validation MAE. The forward pass runs in
bfloat16 (loss and optimiser in fp32), which halves attention memory and roughly doubles throughput. Batches are gathered from
zero-copy sliding windows so the whole dataset never needs to be materialised.

In [ ]:
def batches(W, tod, dow, idx, T_in, bs, mu, sd, shuffle, rng=None):
    if shuffle:
        idx = rng.permutation(idx)
    for i in range(0, len(idx), bs):
        j = np.sort(idx[i:i + bs])
        w = torch.from_numpy(W[j]).to(DEVICE, non_blocking=True)
        steps = j[:, None] + np.arange(T_in)[None]
        x = (w[:, :T_in] - mu) / sd
        yield x, w[:, T_in:], torch.from_numpy(tod[steps]).to(DEVICE), torch.from_numpy(dow[steps]).to(DEVICE)


@torch.no_grad()
def predict(model, W, tod, dow, idx, T_in, bs, mu, sd):
    model.eval()
    out = []
    for x, _, t, d in batches(W, tod, dow, idx, T_in, bs, mu, sd, False):
        with torch.autocast(**AUTOCAST):
            out.append(model(x, t, d).float() * sd + mu)
    return torch.cat(out).cpu().numpy()


def run(name, use_delay, epochs, bs, patience=10, lr=1e-3):
    tag = "PDFormer" if use_delay else "PDFormer-noDelay"
    cfg = DATASETS[name]
    T_in, T_out, spd = cfg["T_in"], cfg["T_out"], 1440 // cfg["step_min"]
    X, adj, times = load_dataset(name)
    tod, dow = time_features(times, cfg["step_min"])
    W = windows(X, T_in, T_out)
    tr, va, te = split_indices(len(W), cfg["split"])
    if MAX_TRAIN_WINDOWS:
        tr = tr[-MAX_TRAIN_WINDOWS:]
    graph = build_graph_inputs(name, X, adj, tr[-1] + T_in, spd)
    mu, sd = graph["mu"], graph["sd"]
    model = PDFormer(X.shape[1], X.shape[2], T_in, T_out, spd, graph, use_delay=use_delay).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    steps_per_epoch = -(-len(tr) // bs)
    warmup = max(1, min(5, epochs // 6)) * steps_per_epoch
    total = epochs * steps_per_epoch
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / warmup if s < warmup else 0.1 + 0.9 * 0.5 * (1 + np.cos(np.pi * (s - warmup) / max(1, total - warmup))))
    loss_fn = nn.HuberLoss(delta=2.0)
    rng = np.random.default_rng(0)
    y_va = W[va, T_in:]
    best, best_state, bad, curve = np.inf, None, 0, []
    if DEVICE == "cuda":
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()   # per-run peak memory, not cumulative
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0.0
        for x, y, t, d in batches(W, tod, dow, tr, T_in, bs, mu, sd, True, rng):
            with torch.autocast(**AUTOCAST):
                pred = model(x, t, d)
            loss = loss_fn(pred.float() * sd + mu, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step(); tot += loss.item() * len(x)
        val = masked_metrics(predict(model, W, tod, dow, va, T_in, bs, mu, sd), y_va, cfg["min_true"])
        curve.append(dict(dataset=name, model=tag, epoch=epoch, train_loss=tot / len(tr), val_MAE=val["MAE"], val_RMSE=val["RMSE"], seconds=time.time() - t0))
        print(f"  {tag} {name} epoch {epoch:3d}  train {tot / len(tr):7.3f}  val MAE {val['MAE']:6.3f}  ({time.time() - t0:.0f}s)", flush=True)
        if val["MAE"] < best:
            best, bad, best_state = val["MAE"], 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif (bad := bad + 1) >= patience:
            print("  early stop"); break
    model.load_state_dict(best_state)
    (DATA / "checkpoints").mkdir(exist_ok=True)
    torch.save(best_state, DATA / "checkpoints" / f"{name}_{tag}.pt")
    p_te = predict(model, W, tod, dow, te, T_in, bs, mu, sd)
    m = metrics_by_horizon(p_te, W[te, T_in:], cfg["min_true"])
    m.insert(0, "model", tag); m.insert(0, "dataset", name)
    eff = dict(dataset=name, model=tag, params=n_params, epochs_run=len(curve), sec_per_epoch=np.mean([c["seconds"] for c in curve]),
               peak_gpu_GB=torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else np.nan, batch_size=bs)
    sample = sample_frame(times, te, T_in, T_out, sample_nodes(adj, X), W[te, T_in:], [p_te], 2, cfg["step_min"], [tag])
    return m, pd.DataFrame(curve), eff, sample

## 4. Experiments

Full PDFormer on all four datasets, then the same model with the delay-aware module removed
(`use_delay=False`) — the ablation that isolates the effect of modelling propagation delay. The epoch
budget below was set from a timing run on the RTX 4090 used for this project; `DMC_EPOCHS=1` (and
`DMC_DATASETS=PEMS08`) turn the whole notebook into a two-minute smoke test, which is what the Docker
image runs by default.

In [ ]:
EPOCHS = {"PEMS04": 30, "PEMS08": 30, "PEMS07": 15, "NYCTaxi": 30}
BATCH = {"PEMS04": 16, "PEMS08": 16, "PEMS07": 8, "NYCTaxi": 16}
ABLATION_ON = ["PEMS04", "PEMS08", "NYCTaxi"]          # PeMS07 (883 sensors) is skipped for time
epochs_override = int(os.environ.get("DMC_EPOCHS", 0)) or None
MAX_TRAIN_WINDOWS = 2000 if epochs_override == 1 else None   # smoke test: a slice of the training set is enough to prove the plumbing

runs = [(name, True) for name in SELECTED] + [(name, False) for name in SELECTED if name in ABLATION_ON]
all_metrics, all_curves, all_eff, all_samples = [], [], [], {}
for name, use_delay in runs:
    t0 = time.time()
    m, curve, eff, sample = run(name, use_delay, epochs_override or EPOCHS[name], BATCH[name])
    all_metrics.append(m); all_curves.append(curve); all_eff.append(eff)
    all_samples.setdefault(name, []).append(sample)
    print(f"{name} {'PDFormer' if use_delay else 'no-delay'}: test MAE {m.query('horizon == "avg"').MAE.item():.3f}  ({(time.time() - t0) / 60:.1f} min)")

metrics = pd.concat(all_metrics, ignore_index=True); metrics.to_csv(RESULTS / "metrics_pdformer.csv", index=False)
pd.concat(all_curves, ignore_index=True).to_csv(RESULTS / "train_curves_pdformer.csv", index=False)
pd.DataFrame(all_eff).to_csv(RESULTS / "efficiency_pdformer.csv", index=False)
for name, frames in all_samples.items():
    out = frames[0]
    for f in frames[1:]:
        out = out.merge(f.drop(columns=["actual_h1", "actual_hL"]), on=["time_h1", "time_hL", "node"])
    out.to_csv(RESULTS / f"preds_sample_{name}_pdformer.csv", index=False)

## 5. Results against the traditional models (average over horizons, test set)

In [ ]:
tradi = pd.read_csv(RESULTS / "metrics_tradi.csv") if (RESULTS / "metrics_tradi.csv").exists() else pd.DataFrame(columns=metrics.columns)
both = pd.concat([tradi, metrics], ignore_index=True)
both = both[both.dataset.isin(SELECTED)]
table = both.query("horizon == 'avg'").pivot(index="model", columns="dataset", values=["MAE", "RMSE", "MAPE"]).round(2)
table = table.reindex([m for m in ["Persistence", "HistoricalAverage", "Ridge", "PDFormer-noDelay", "PDFormer"] if m in table.index])
table

In [ ]:
paper = pd.DataFrame({"PEMS04": [18.32, 29.97, 12.10], "PEMS08": [13.58, 23.51, 9.05], "PEMS07": [19.83, 32.87, 8.53], "NYCTaxi": [np.nan] * 3},
                     index=["MAE", "RMSE", "MAPE"]).T
ours = metrics.query("model == 'PDFormer' and horizon == 'avg'").set_index("dataset")[["MAE", "RMSE", "MAPE"]]
compare = ours.join(paper, rsuffix="_paper").round(2)
compare["MAE_gap_%"] = (100 * (compare.MAE / compare.MAE_paper - 1)).round(1)
compare

The paper trains for 200 epochs with 6 layers; the `MAE_gap_%` column is how far this 4-layer, ≤ 30-epoch
version lands from the published numbers (NYCTaxi is reported per channel in the paper, so no direct
comparison). Per-horizon curves, training curves and the prediction plots are in `dashboard.ipynb`.

In [ ]:
multi = [n for n in SELECTED if DATASETS[n]["T_out"] > 1]   # a one-step dataset has no horizon curve
fig, axes = plt.subplots(1, len(multi), figsize=(5 * len(multi), 4), squeeze=False)
for ax, name in zip(axes[0], multi):
    d = both.query("dataset == @name and horizon != 'avg'").astype({"horizon": int})
    for model, g in d.groupby("model"):
        ax.plot(g.horizon, g.MAE, marker="o", label=model)
    ax.set_title(f"{name}: MAE by horizon"); ax.set_xlabel(f"steps ahead (x {DATASETS[name]['step_min']} min)"); ax.legend()
plt.tight_layout()

## 6. Does modelling propagation delay help? (ablation)

In [ ]:
abl = metrics.query("horizon == 'avg'").pivot(index="dataset", columns="model", values="MAE")
if "PDFormer-noDelay" in abl:
    abl["delay_gain_%"] = (100 * (1 - abl["PDFormer"] / abl["PDFormer-noDelay"])).round(2)
    abl.to_csv(RESULTS / "ablation_delay.csv")
abl.round(3)

A positive `delay_gain_%` means the delay-aware feature transformation lowered the test MAE. Combined
with the cross-correlation analysis in `dataset_analysis.ipynb`, the write-up in
`propagation_delay_findings.md` discusses where the delay signal exists (5-minute highway sensors) and
where it does not (30-minute taxi grid), and what that means for the module.

## 7. Efficiency

In [ ]:
pd.DataFrame(all_eff).round(3)

## 8. Take-aways

- A four-layer PDFormer with ~0.4 M parameters beats every traditional baseline on every dataset, and the
  margin grows with the horizon — exactly where Ridge and Persistence fall apart and where the spatial
  (graph) context matters.
- The cost is real: minutes per epoch on a data-centre GPU versus seconds for Ridge on a CPU. For a
  5-minute-ahead alert the linear model is arguably the better production choice; for the 30–60 minute
  planning horizon the transformer earns its keep.
- Attention over 883 sensors is the memory bottleneck (`peak_gpu_GB` above): every head materialises an
  N × N map per time step, which is why PeMS07 runs at half the batch size and was left out of the ablation.